# 03 - Train and compare classical audio classifiers

This notebook trains a Linear Support Vector Machine and a Random Forest using the MFCC features created in notebook 02. It uses the training split for fitting and the validation split for model selection. The test split is deliberately held back for the final evaluation stage.

Both models use class weighting to reduce the effect of the uneven class distribution.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import (
    train_and_evaluate,
    evaluate_classifier,
    save_model_bundle,
    save_metrics,
)

DATA_DIR = PROJECT_ROOT / "notebooks" / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_PATH = DATA_DIR / "mfcc_features.npz"
if not FEATURE_PATH.is_file():
    raise FileNotFoundError("MFCC feature file missing. Run notebooks 01 and 02 first.")

print("Project root:", PROJECT_ROOT)
print("Feature archive:", FEATURE_PATH)

## 2. Load the prepared train, validation and test arrays

In [ ]:
with np.load(FEATURE_PATH, allow_pickle=False) as data:
    X_train = data["X_train"].astype(np.float32)
    y_train = data["y_train"].astype(str)
    X_validation = data["X_validation"].astype(np.float32)
    y_validation = data["y_validation"].astype(str)
    X_test = data["X_test"].astype(np.float32)
    y_test = data["y_test"].astype(str)
    feature_names = data["feature_names"].astype(str)

print("Training:", X_train.shape, pd.Series(y_train).value_counts().to_dict())
print("Validation:", X_validation.shape)
print("Test (held back):", X_test.shape)
assert X_train.shape[1] == X_validation.shape[1] == X_test.shape[1] == len(feature_names)
assert np.isfinite(X_train).all() and np.isfinite(X_validation).all() and np.isfinite(X_test).all()

## 3. Train the two baseline models

In [ ]:
fitted_models, comparison = train_and_evaluate(
    X_train, y_train, X_validation, y_validation, random_state=42
)
display(comparison.round(4))

metrics_path = save_metrics(comparison, MODEL_DIR / "validation_model_comparison.json")
print("Saved validation comparison:", metrics_path)

## 4. Inspect per-class performance

In [ ]:
validation_results = {}
for model_name, model in fitted_models.items():
    result = evaluate_classifier(model, X_validation, y_validation)
    validation_results[model_name] = result
    print(f"\\n{model_name} per-class report")
    display(pd.DataFrame(result["report"]).T.round(3))

    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(
        confusion_matrix=result["confusion_matrix"],
        display_labels=result["labels"],
    ).plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
    ax.set_title(f"{model_name.replace('_', ' ').title()} - validation confusion matrix")
    plt.tight_layout()
    plt.show()

## 5. Save the best validation model

In [ ]:
best_model_name = str(comparison.iloc[0]["model"])
best_model = fitted_models[best_model_name]
bundle_path = save_model_bundle(
    best_model,
    MODEL_DIR / f"{best_model_name}.joblib",
    model_name=best_model_name,
    feature_names=feature_names,
)
print("Best validation model:", best_model_name)
print("Saved model bundle:", bundle_path)
print("Validation macro-F1:", round(float(comparison.iloc[0]["macro_f1"]), 4))

print("\\nImportant: the test split has not been used for model selection.")

## 6. What to record from this experiment

Compare macro-F1 and balanced accuracy rather than relying on accuracy alone. Macro-F1 gives every command class equal weight, which matters because `unknown` has many more examples than most individual commands. Review the confusion matrices to see which commands are commonly mixed up. Training and validation prediction times provide an initial efficiency comparison.

The saved model is a baseline artifact. The CNN experiment will be added separately, and the final test set will be used only after model selection is complete.